In [ ]:
# prompt: Install sympy
# numpy and drawsvg

%pip install sympy numpy drawsvg matplotlib scipy ipywidgets


# Light Guide Panel Generator

Light Guide Panels are the (literal) backbone of LCD screen illumination.
They're the diffusion secret sauce behind uniformly backlit LCD displays.
When illuminated from the edge, they illuminate outwards through the pattern etched on the surface of the arylic.

It turns out that design software for this sort of thing [already exists](https://www.febees.com/index.html). But why pay \$1500 when we can write a crude version that will do the job?

In [ ]:
#%matplotlib inline
import matplotlib.pyplot as plt
import drawsvg as draw
import numpy as np
from scipy.optimize import curve_fit
import scipy.interpolate as interpolate
from ipywidgets import interactive
from IPython.display import SVG, display, HTML
import base64


# Extra fn for rescaling displayed SVGs
_html_template='<img width="{}" src="data:image/svg+xml;base64,{}" >'

# def svg_to_fixed_width_html_image(svg, width="100%"):
#     text = _html_template.format(width, base64.b64encode(svg))
#     return HTML(text)

def svg_to_fixed_width_html_image(svg, width="50%"):
    b64 = base64.b64encode(svg).decode("utf8")
    text = f'<img width="{width}" src="data:image/svg+xml;base64,{b64}" >'
    return HTML(text)
import random

import os
from datetime import datetime

def get_timestamped_filename(style_name):
    output_dir = "generated"
    os.makedirs(output_dir, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    return os.path.join(output_dir, f"{timestamp}_{style_name}.svg")

def cap_value(value, min_value, max_value):
    return max(min_value, min(value, max_value))


def create_thick_bezier_path(p0, p1, p2, width=0.5, num_samples=30, fill_color='#00FF00', cap='round'):
    """
    生成在激光切割/雕刻软件中具有物理宽度的闭合弧形路径（闭合 2D 填充图形）。
    端点处理(cap):
    - 'round': 突出凸起的半圆弧端头 (convex round cap)
    - 'flat': 平直直线端头 (straight line cap)
    """
    P0 = np.array(p0, dtype=float)
    P1 = np.array(p1, dtype=float)
    P2 = np.array(p2, dtype=float)
    
    t_vals = np.linspace(0, 1, num_samples)
    r = width / 2.0
    
    left_pts = []
    right_pts = []
    
    for t in t_vals:
        pt = (1-t)**2 * P0 + 2*(1-t)*t * P1 + t**2 * P2
        tan = 2*(1-t)*(P1 - P0) + 2*t*(P2 - P1)
        norm = np.array([-tan[1], tan[0]])
        norm_len = np.linalg.norm(norm)
        if norm_len > 0:
            norm = norm / norm_len
        left_pts.append(pt + r * norm)
        right_pts.append(pt - r * norm)
        
    path = draw.Path(fill=fill_color, stroke='none')
    # 1. 沿左侧外边界前进
    path.M(left_pts[0][0], left_pts[0][1])
    for pt in left_pts[1:]:
        path.L(pt[0], pt[1])
        
    # 2. 终点端头 (cap)：向外突出的半圆弧或直平线
    if cap == 'round':
        path.A(r, r, 0, 0, 0, right_pts[-1][0], right_pts[-1][1])
    else:
        path.L(right_pts[-1][0], right_pts[-1][1])
        
    # 3. 沿右侧外边界返回
    for pt in reversed(right_pts[:-1]):
        path.L(pt[0], pt[1])
        
    # 4. 起点端头 (cap)：向外突出的半圆弧或直平线
    if cap == 'round':
        path.A(r, r, 0, 0, 0, left_pts[0][0], left_pts[0][1])
    else:
        path.L(left_pts[0][0], left_pts[0][1])
        
    path.Z()
    return path


def create_thick_polyline_path(points, width=0.5, fill_color='#00FF00', cap='round'):
    """
    将折线/点集（如圆环上的弧线点）转换为具有物理宽度的闭合 2D 填充图形。
    - points: [(x0, y0), (x1, y1), ...] 沿中心线的点坐标列表
    - width: 弧线宽度 (mm)
    - fill_color: 填充颜色
    - cap: 端头形状 ('round' 为外凸半圆弧，'flat' 为直线)
    """
    pts = np.array(points, dtype=float)
    n = len(pts)
    if n < 2:
        return None
        
    r = width / 2.0
    
    tangents = []
    normals = []
    for i in range(n - 1):
        t = pts[i+1] - pts[i]
        norm_t = np.linalg.norm(t)
        if norm_t == 0:
            t = np.array([1.0, 0.0])
        else:
            t = t / norm_t
        tangents.append(t)
        n_vec = np.array([-t[1], t[0]])
        normals.append(n_vec)
        
    left_pts = []
    right_pts = []
    
    left_pts.append(pts[0] + r * normals[0])
    right_pts.append(pts[0] - r * normals[0])
    
    for i in range(1, n - 1):
        v_norm = normals[i-1] + normals[i]
        len_vn = np.linalg.norm(v_norm)
        if len_vn > 0:
            v_norm = v_norm / len_vn
        else:
            v_norm = normals[i]
        left_pts.append(pts[i] + r * v_norm)
        right_pts.append(pts[i] - r * v_norm)
        
    left_pts.append(pts[-1] + r * normals[-1])
    right_pts.append(pts[-1] - r * normals[-1])
    
    path = draw.Path(fill=fill_color, stroke='none')
    path.M(left_pts[0][0], left_pts[0][1])
    for pt in left_pts[1:]:
        path.L(pt[0], pt[1])
        
    if cap == 'round':
        path.A(r, r, 0, 0, 0, right_pts[-1][0], right_pts[-1][1])
    else:
        path.L(right_pts[-1][0], right_pts[-1][1])
        
    for pt in reversed(right_pts[:-1]):
        path.L(pt[0], pt[1])
        
    if cap == 'round':
        path.A(r, r, 0, 0, 0, left_pts[0][0], left_pts[0][1])
    else:
        path.L(left_pts[0][0], left_pts[0][1])
        
    path.Z()
    return path


def create_thick_circular_arc(r_center, theta_start, theta_end, width=0.5, fill_color='#00FF00', cap='round'):
    """
    在极坐标圆环上绘制具有物理宽度的同心圆弧 (Concentric Circular Arc Ribbon)。
    - r_center: 弧线中心半径 (mm)
    - theta_start, theta_end: 弧线起始和终止角度 (radians)
    - width: 弧线宽度 (mm)
    - fill_color: 填充颜色
    - cap: 端头形状 ('round' 为外凸半圆弧，'flat' 为直线)
    """
    r_in = r_center - width / 2.0
    r_out = r_center + width / 2.0
    r_cap = width / 2.0
    
    if theta_start > theta_end:
        theta_start, theta_end = theta_end, theta_start
        
    start_in = (r_in * np.cos(theta_start), r_in * np.sin(theta_start))
    end_in = (r_in * np.cos(theta_end), r_in * np.sin(theta_end))
    start_out = (r_out * np.cos(theta_start), r_out * np.sin(theta_start))
    end_out = (r_out * np.cos(theta_end), r_out * np.sin(theta_end))
    
    path = draw.Path(fill=fill_color, stroke='none')
    path.M(start_in[0], start_in[1])
    
    angle_diff = theta_end - theta_start
    large_arc = 1 if angle_diff > np.pi else 0
    
    # 1. 沿内侧圆弧绘制到终点
    path.A(r_in, r_in, 0, large_arc, 1, end_in[0], end_in[1])
    
    # 2. 终点端头 (cap)
    if cap == 'round':
        path.A(r_cap, r_cap, 0, 0, 0, end_out[0], end_out[1])
    else:
        path.L(end_out[0], end_out[1])
        
    # 3. 沿外侧圆弧返回起点
    path.A(r_out, r_out, 0, large_arc, 0, start_out[0], start_out[1])
    
    # 4. 起点端头 (cap)
    if cap == 'round':
        path.A(r_cap, r_cap, 0, 0, 0, start_in[0], start_in[1])
    else:
        path.L(start_in[0], start_in[1])
        
    path.Z()
    return path


In [ ]:
import ipywidgets as widgets
from IPython.display import display

# R_OUTER = 263/2
# R_INNER = 104/2

R_OUTER = 26/2
R_INNER = 10/2

INNER_OFFSET_MM = 1.0  # Distance from inner red circle (mm)
OUTER_OFFSET_MM = 1.0  # Distance from outer red circle (mm)

num_points_slider = widgets.IntSlider(
    value=70,
    min=5,
    max=100,
    step=5,
    description='Num Points:',
    continuous_update=False
)
display(num_points_slider)

In [ ]:
HALF_WIDTH_MM = (R_OUTER - R_INNER) / 2
HEIGHT_MM = (R_OUTER + R_INNER) * np.pi
NOMINAL_DOTS_PER_MM = 1.0

# Scale DOT_DIAMETER_MM dynamically according to panel size R_OUTER
scale_factor = R_OUTER / 60.0
BASE_DOT_DIAMETER_MM = 0.5
DOT_DIAMETER_MM = max(0.2, round(BASE_DOT_DIAMETER_MM * scale_factor, 3))
CORNER_RADIUS_MM = 0

HEIGHT_MM = round(HEIGHT_MM, 1)
print(f"Half Width: {HALF_WIDTH_MM}")
print(f"Height: {HEIGHT_MM}")
print(f"Dynamic DOT_DIAMETER_MM: {DOT_DIAMETER_MM} mm")


We can create a spline function with handles that can help us model dot density as a function of distance from the light source.

In [ ]:
import ipywidgets as widgets

def spline_density_gen(y0=1.0, y1=1.0, y2=1.0, y3=1.0, y4=1.0):
    # 5 control points fixed at 25% horizontal intervals (0%, 25%, 50%, 75%, 100%)
    x = np.array([0.0, HALF_WIDTH_MM * 0.25, HALF_WIDTH_MM * 0.50, HALF_WIDTH_MM * 0.75, HALF_WIDTH_MM])
    y = np.array([y0, y1, y2, y3, y4])
    
    t, c, k = interpolate.splrep(x, y, s=0, k=2)
    spline = interpolate.BSpline(t, c, k, extrapolate=False)

    plt.figure(figsize=(10, 4.5))
    N = 200
    xx = np.linspace(0, HALF_WIDTH_MM, N)
    yy = spline(xx)

    plt.plot(xx, yy, 'r-', linewidth=2, label='BSpline Density')
    plt.fill_between(xx, 0, yy, color='red', alpha=0.1)
    plt.plot(x, y, 'bo', markersize=6, label='Control Points')
    
    for px, py in zip(x, y):
        plt.annotate(f" ({px:.2f}, {py:.2f})", xy=(px, py), fontsize=9)

    plt.axhline(1/NOMINAL_DOTS_PER_MM, color='gray', linestyle='--', label=f'Nominal ({1/NOMINAL_DOTS_PER_MM:.2f})')
    
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.legend(loc='upper right')
    plt.title("Half Length Illumination Density Profile", fontsize=12, fontweight='bold')
    plt.ylabel("Dot Density [dots/mm]")
    plt.xlabel("Distance from Center [mm]")
    plt.ylim(bottom=0)
    plt.xlim(0, HALF_WIDTH_MM)
    plt.tight_layout()
    plt.show()
    return t, c, k

# Styled vertical sliders for density at 5 control points (0%, 25%, 50%, 75%, 100%)
max_density = round(2.0 / NOMINAL_DOTS_PER_MM, 2)

y0_slider = widgets.FloatSlider(value=0.97, min=0, max=max_density, step=0.01, description='Center (y0 0%):', continuous_update=False)
y1_slider = widgets.FloatSlider(value=1.0, min=0, max=max_density, step=0.01, description='Pos 1 (y1 25%):', continuous_update=False)
y2_slider = widgets.FloatSlider(value=1.0, min=0, max=max_density, step=0.01, description='Pos 2 (y2 50%):', continuous_update=False)
y3_slider = widgets.FloatSlider(value=1.0, min=0, max=max_density, step=0.01, description='Pos 3 (y3 75%):', continuous_update=False)
y4_slider = widgets.FloatSlider(value=1.09, min=0, max=max_density, step=0.01, description='Edge (y4 100%):', continuous_update=False)

i_sdf = widgets.interactive(
    spline_density_gen,
    y0=y0_slider,
    y1=y1_slider,
    y2=y2_slider,
    y3=y3_slider,
    y4=y4_slider
)

display(i_sdf)


In [ ]:
dot_step_factor = 1 # how far apart dots should be

In [ ]:
# Create the spline function from the density parameters above.
rho = interpolate.BSpline(*i_sdf.result, extrapolate=False)
def dotspace(x0, xf, endpoint=False):
    """return an array of dot locations from x0 to xf spaced
        according to the dot density profile."""
    dots_x = []
    x = x0
    while x < xf:
        dots_x.append(x)
        dots_per_mm_x = rho(x) / dot_step_factor
        print(x, dots_per_mm_x)
        x += 1/dots_per_mm_x  # Step forward mm-per-dot
    return np.array(dots_x)

In [ ]:

Y = np.arange(0, HEIGHT_MM, NOMINAL_DOTS_PER_MM*2)[:-1]
X = dotspace(0, HALF_WIDTH_MM - NOMINAL_DOTS_PER_MM)
xx, yy = np.meshgrid(X, Y) # create HALF of the x and y index of all the points given their x and y separately in rows and columns

print(X)
print(dotspace(0, HALF_WIDTH_MM - NOMINAL_DOTS_PER_MM))
yoffset = (Y[1] - Y[0])/2 # to make the lines centered, because we set the canvas origin horizontally centered, to we only need to calculate the top offset
xoffset = (X[1] - X[0])/2 # to make the alternate lines at the center of the other lines
Y2 = np.arange(yoffset, HEIGHT_MM+yoffset, NOMINAL_DOTS_PER_MM*2)[:-1]
X2 = dotspace(xoffset, HALF_WIDTH_MM - NOMINAL_DOTS_PER_MM)
xx2, yy2 = np.meshgrid(X2,Y2) # this creates the other, offsetted HALF

# Move non-illuminated sides off the top/bottom edge.
miny = min(yy[0][0], yy2[0][0])
maxy = max(yy[-1][-1], yy2[-1][-1])
deltay = HEIGHT_MM - (maxy-miny)
yy += deltay/2
yy2 += deltay/2

# TODO: we need to renormalize the output since it has been stretched beyond the width of the original img.

# Preview Plot for sanity checking:
# fig = plt.figure(figsize=(15,15))
# ax = fig.add_subplot(111)
# ax.set_aspect('equal')
# ax.plot(xx, yy, ls="None", marker=".", color="blue")
# ax.plot(xx2, yy2, ls="None", marker=".", color="green")
# ax.plot(-xx, yy, ls="None", marker=".", color="blue")
# ax.plot(-xx2, yy2, ls="None", marker=".", color="green")
# plt.tight_layout()
# plt.show()

Draw the SVG from the above parameters.

In [ ]:
def translate_to_polar_offset(x, y, y_max, xoffset=0, inner_offset=INNER_OFFSET_MM, outer_offset=OUTER_OFFSET_MM):
    r_min = R_INNER + inner_offset
    r_max = R_OUTER - outer_offset
    r = cap_value(x + xoffset, r_min, r_max)
    y_temp = (np.pi / y_max) * y
    # x' = r * cos(theta)
    polar_x = r * np.cos(y_temp)
    # y' = r * sin(theta)
    polar_y = r * np.sin(y_temp)
    return polar_x, polar_y

In [ ]:
def prepare_points(x, y):
  # x_new = x + square_width / 2
  # y_new = y - square_height / 2
  x_new = x + HALF_WIDTH_MM
  y_new = y - HEIGHT_MM / 2
  return x_new, y_new

In [ ]:
# Create an SVG replacing the two meshgrids from above with actual circles.
offset = (10, 10)
fill_color = 'black'

# d = draw.Drawing(2*HALF_WIDTH_MM + 2*offset[0],
#                  HEIGHT_MM + 2*offset[0],
#                  origin=(-HALF_WIDTH_MM - offset[0], 0),
#                  displayInline=False)
# d3.width = f"{2*HALF_WIDTH_MM + 2*offset[0]}mm"
# d3.height = f"{HEIGHT_MM + 2*offset[1]}mm"
# d.set_pixel_scale(1)
# r = draw.Rectangle(-HALF_WIDTH_MM, 0,
#                    2*HALF_WIDTH_MM, HEIGHT_MM,
#                    rx=CORNER_RADIUS_MM,
#                    fill='#FFFFFF', stroke='red', stroke_width="0.1")
# d.append(r)
# # Produce the circle pattern.
# # Skip duplicate circles to avoid weird bug where LaserCAD software skips etching anything
# # if there are copies at the same location.
# #for xx, yy, fill_color in [[xx, yy, 'red'], [xx2, yy2, 'black'], [-xx[:,1:], yy[:,1:], 'green'], [-xx2, yy2, 'black']]:
# for xx, yy in [[xx, yy], [xx2, yy2], [-xx[:,1:], yy[:,1:]], [-xx2, yy2]]:
#     for x,y in zip(xx.flatten(), yy.flatten()):
#         c = draw.Circle(x, y, r=DOT_DIAMETER_MM/2.,
#                         stroke_width='0.1', stroke=fill_color,
#                         fill_opacity=1.0)#, id='circle')
#         d.append(c)
# d.save_svg('example.svg')

d3 = draw.Drawing(2*HALF_WIDTH_MM + 2*offset[0],
                 HEIGHT_MM + 2*offset[1],
                 origin=(0, -HEIGHT_MM / 2 - offset[1]),
                 displayInline=False)
# d3.width = f"{2*HALF_WIDTH_MM + 2*offset[0]}mm"
# d3.height = f"{HEIGHT_MM + 2*offset[1]}mm"
d3.set_pixel_scale(1)
r = draw.Rectangle(0, -HEIGHT_MM / 2,
                   2*HALF_WIDTH_MM, HEIGHT_MM,
                   rx=CORNER_RADIUS_MM,
                   fill='#FFFFFF', stroke='red', stroke_width="0.1")
d3.append(r)

# Produce the circle pattern.
# Skip duplicate circles to avoid weird bug where LaserCAD software skips etching anything
# if there are copies at the same location.
#for xx, yy, fill_color in [[xx, yy, 'red'], [xx2, yy2, 'black'], [-xx[:,1:], yy[:,1:], 'green'], [-xx2, yy2, 'black']]:
for xx, yy in [[xx, yy], [xx2, yy2], [-xx[:,1:], yy[:,1:]], [-xx2, yy2]]:
    for x,y in zip(xx.flatten(), yy.flatten()):

        x, y = prepare_points(x, y)
        delta_left = random.uniform(DOT_DIAMETER_MM / 4, DOT_DIAMETER_MM / 2)
        delta_right = random.uniform(DOT_DIAMETER_MM / 4, DOT_DIAMETER_MM / 2)
        # 3 points:
        #  center point (x, y)
        #  bottom point (x, y - d_left)
        #  top point    (x, y + d_right)
        c = draw.Line(x, y - delta_left, x, y + delta_right, stroke=fill_color, stroke_width="0.05")
        d3.append(c)

x_axis = draw.Line(0, 0, 2*HALF_WIDTH_MM, 0, stroke='black', stroke_width="0.1")
d3.append(x_axis)
print(HEIGHT_MM)
y_axis = draw.Line(0, 0, 0, HEIGHT_MM / 2, stroke='blue', stroke_width="0.1")
d3.append(y_axis)

filename_moved = get_timestamped_filename('moved')
d3.save_svg(filename_moved)


In [ ]:
#display(SVG(url='https://upload.wikimedia.org/wikipedia/commons/f/f6/People_%28example%29.svg'))
#svg_to_fixed_width_html_image(SVG(filename_moved).data.encode('ascii'), width="100%")

In [ ]:
d2 = draw.Drawing(2 * R_OUTER + 2 * offset[0],
                  2 * R_OUTER + 2 * offset[1],
                  origin = (-R_OUTER - offset[0], -R_OUTER - offset[1]),
                  displayInline=False)
d2.width = f"{2*R_OUTER + 2*offset[0]}mm"
d2.height = f"{2*R_OUTER + 2*offset[1]}mm"
d2.set_pixel_scale(1)
r2 = draw.Circle(0, 0, r=R_OUTER, fill='#FFFFFF', stroke='red', stroke_width="0.1")
d2.append(r2)
r3 = draw.Circle(0, 0, r=R_INNER, fill='#FFFFFF', stroke='red', stroke_width="0.1")
d2.append(r3)

for xx, yy in [[xx, yy], [xx2, yy2], [-xx[:,1:], yy[:,1:]], [-xx2, yy2]]:
    for x,y in zip(xx.flatten(), yy.flatten()):
        x, y = prepare_points(x, y)

        delta_left = random.uniform(DOT_DIAMETER_MM * 1, DOT_DIAMETER_MM * 2)
        delta_right = random.uniform(DOT_DIAMETER_MM * 1, DOT_DIAMETER_MM * 2)

        polar_line_x, polar_line_y = translate_to_polar_offset(x, y + delta_left, HEIGHT_MM / 2, R_INNER)

        polar_end_x, polar_end_y, = translate_to_polar_offset(x, y - delta_right, HEIGHT_MM / 2, R_INNER)

        c = draw.Line(polar_line_x, polar_line_y,
                           polar_end_x, polar_end_y,
                           stroke=fill_color, stroke_width="0.05")
        # print(x, y, polar_line_x)
        d2.append(c)

filename_polar = get_timestamped_filename('polar')
d2.save_svg(filename_polar)


In [ ]:
#display(SVG(url='https://upload.wikimedia.org/wikipedia/commons/f/f6/People_%28example%29.svg'))
#svg_to_fixed_width_html_image(SVG(filename_polar).data.encode('ascii'), width="100%")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import UnivariateSpline
import ipywidgets as widgets
from IPython.display import display

def visualize_spline_with_randomized_dots_2d(spline_func, x_range, y_range, num_x_areas, num_y_areas, num_points_total):
    """
    Visualize a spline equation and overlay randomized dots with density proportional to y values.
    The plot is divided into areas along both x and y axes.
    """
    x = np.linspace(x_range[0], x_range[1], 1000)
    y = spline_func(x)
    y = np.nan_to_num(y, nan=0.0)

    y_min, y_max = np.min(y), np.max(y)
    denom = y_max - y_min
    if denom <= 0:
        y_normalized = np.zeros_like(y)
    else:
        y_normalized = (y - y_min) / denom

    sum_y = np.sum(y_normalized)
    if sum_y <= 0:
        y_weights = np.zeros_like(y_normalized)
    else:
        y_weights = y_normalized / sum_y

    x_boundaries = np.linspace(x_range[0], x_range[1], num_x_areas + 1)
    y_boundaries = np.linspace(y_range[0], y_range[1], num_y_areas + 1)

    plt.figure(figsize=(10, 6))
    plt.plot(x, y, label='Spline Density Profile', color='blue', lw=2)

    dots = []
    base_dots_per_area = 0
    for i in range(num_x_areas):
        for j in range(num_y_areas):
            x_min, x_max = x_boundaries[i], x_boundaries[i + 1]
            y_min_b, y_max_b = y_boundaries[j], y_boundaries[j + 1]

            mask_x = (x >= x_min) & (x < x_max)
            x_in_area = x[mask_x]
            y_in_area = y[mask_x]

            if len(y_in_area) > 0:
                area_weight = np.sum(y_weights[mask_x])
            else:
                area_weight = 0.0

            if np.isnan(area_weight):
                area_weight = 0.0

            weighted_dots = int(np.nan_to_num(num_points_total * area_weight, nan=0))
            num_dots = max(base_dots_per_area, weighted_dots)

            x_dots = np.random.uniform(x_min, x_max, num_dots)
            y_dots = np.random.uniform(y_min_b, y_max_b, num_dots)

            dots.extend(zip(x_dots, y_dots))
            plt.scatter(x_dots, y_dots, color='red', s=10, alpha=0.6, label=None)

    plt.title(f'Spline Visualization with Randomized Dots (Total Points: {len(dots)})')
    plt.xlabel('X')
    plt.ylabel('Y')
    plt.legend()
    plt.grid(True)
    plt.show()

    return dots

if 'num_points_slider' in globals():
    num_pts = num_points_slider.value
else:
    num_pts = 1000

engrave_dots = visualize_spline_with_randomized_dots_2d(
    rho,
    x_range=(0, HALF_WIDTH_MM),
    y_range=(0, 3),
    num_x_areas=5,
    num_y_areas=4,
    num_points_total=num_pts
)

print(f"Generated {len(engrave_dots)} dots using Num Points ({num_pts}).")


In [ ]:
def scale_and_move_points(points, from_x, from_y, to_x, to_y):
    scaled_points = []

    x_scale = to_x / from_x
    y_scale = to_y / from_y

    for x, y in points:
        new_x = x * x_scale
        new_y = y * y_scale - to_y / 2  # Move down by y_scale / 2
        scaled_points.append((new_x, new_y))
    return scaled_points

In [ ]:
# Scale engrave_dots dynamically according to R_OUTER and R_INNER (HALF_WIDTH_MM and HEIGHT_MM)
to_x = 2 * HALF_WIDTH_MM  # R_OUTER - R_INNER
to_y = HEIGHT_MM

enlarged_dots = scale_and_move_points(engrave_dots, from_x=5, from_y=3, to_x=to_x, to_y=to_y)


In [ ]:
import matplotlib.pyplot as plt


# Assuming 'enlarged_dots' is already defined from the previous code

plt.figure(figsize=(8, 8))
x_coords, y_coords = zip(*enlarged_dots)
plt.scatter(x_coords, y_coords, color='red', s=10, alpha=0.6)  # Adjust 's' for dot size
plt.title('Visualizing Enlarged Dots')
plt.xlabel('X')
plt.ylabel('Y')
plt.grid(True)
plt.show()
import random


In [ ]:
ENGRAVE_COLOR = "#00FF00"

In [ ]:
import matplotlib.pyplot as plt
import drawsvg as draw
import numpy as np
from scipy.optimize import curve_fit
import scipy.interpolate as interpolate
from ipywidgets import interactive
from IPython.display import SVG, display, HTML
import base64
from scipy.interpolate import UnivariateSpline


def translate_to_polar_offset(x, y, y_max, xoffset=0, inner_offset=INNER_OFFSET_MM, outer_offset=OUTER_OFFSET_MM):
    r_min = R_INNER + inner_offset
    r_max = R_OUTER - outer_offset
    r = cap_value(x + xoffset, r_min, r_max)
    y_temp = (np.pi / y_max) * y
    # x' = r * cos(theta)
    polar_x = r * np.cos(y_temp)
    # y' = r * sin(theta)
    polar_y = r * np.sin(y_temp)
    return polar_x, polar_y

# Now use enlarged_dots with translate_to_polar_offset in the drawing
d2 = draw.Drawing(2 * R_OUTER + 2 * offset[0],
                  2 * R_OUTER + 2 * offset[1],
                  origin=(-R_OUTER - offset[0], -R_OUTER - offset[1]),
                  displayInline=False)
d2.width = f"{2*R_OUTER + 2*offset[0]}mm"
d2.height = f"{2*R_OUTER + 2*offset[1]}mm"
d2.set_pixel_scale(1)
r2 = draw.Circle(0, 0, r=R_OUTER, fill='none', stroke='red', stroke_width="0.1")
d2.append(r2)
r3 = draw.Circle(0, 0, r=R_INNER, fill='none', stroke='red', stroke_width="0.1")
d2.append(r3)


for x, y in enlarged_dots:

    polar_x, polar_y = translate_to_polar_offset(x, y, 5, R_INNER) # Assuming 10 is the relevant y_max here, adjust if necessary
    c = draw.Circle(polar_x, polar_y, r=DOT_DIAMETER_MM/2, stroke_width='0.1', stroke=ENGRAVE_COLOR, fill_opacity=1.0)
    d2.append(c)


filename_dots = get_timestamped_filename('dots')
d2.save_svg(filename_dots)
svg_to_fixed_width_html_image(SVG(filename_dots).data.encode('ascii'), width="100%")
import random


In [ ]:
%pip install "drawsvg[all]"


In [ ]:
import drawsvg as dw
import numpy as np

def calculate_arc_parameters(p1, p2, p3):
    """
    Calculate the SVG arc parameters to ensure the arc passes through three points.
    Parameters:
    - p1, p2, p3: tuples representing (x, y) coordinates of three points.
    Returns:
    - rx, ry: Radii of the ellipse
    - cx, cy: Center of the ellipse
    - rotation: Rotation of the ellipse (in degrees)
    - large_arc_flag: 0 or 1
    - sweep_flag: 0 or 1
    """
    # Create a matrix for solving the circle's center
    A = np.array([
        [p1[0] - p2[0], p1[1] - p2[1]],
        [p1[0] - p3[0], p1[1] - p3[1]]
    ])
    b = np.array([
        (p1[0]**2 - p2[0]**2 + p1[1]**2 - p2[1]**2) / 2,
        (p1[0]**2 - p3[0]**2 + p1[1]**2 - p3[1]**2) / 2
    ])
    center = np.linalg.solve(A, b)

    # Compute the radius
    radius = np.sqrt((p1[0] - center[0])**2 + (p1[1] - center[1])**2)

    # Set arc parameters
    rx = ry = radius
    rotation = 0  # No rotation for a circle-based arc
    large_arc_flag = 1
    sweep_flag = 1

    return rx, ry, center[0], center[1], rotation, large_arc_flag, sweep_flag


def draw_arc_with_three_points(p1, p2, p3):
    """
    Draw an arc passing through three points using drawsvg.
    """
    rx, ry, cx, cy, rotation, large_arc_flag, sweep_flag = calculate_arc_parameters(p1, p2, p3)

    # Create a drawing canvas
    d = dw.Drawing(500, 500, origin='center')

    # Draw the three points
    d.append(dw.Circle(p1[0], p1[1], 2, fill='red'))
    d.append(dw.Circle(p2[0], p2[1], 2, fill='green'))
    d.append(dw.Circle(p3[0], p3[1], 2, fill='blue'))

    # Draw the arc
    d.append(dw.Path(f'M {p1[0]} {p1[1]} A {rx} {ry} {rotation} {large_arc_flag} {sweep_flag} {p3[0]} {p3[1]}',
                     stroke='black', fill='none'))

    return d

# Example points
p1 = [50, 50]
p2 = [100, 100]
p3 = [150, 50]

# Draw and save the arc
d = draw_arc_with_three_points(p1, p2, p3)

# svg_html = d.as_svg()
# display(HTML(svg_html))

In [ ]:
# Now use enlarged_dots with translate_to_polar_offset in the drawing
d2 = draw.Drawing(2 * R_OUTER + 2 * offset[0],
                  2 * R_OUTER + 2 * offset[1],
                  origin=(-R_OUTER - offset[0], -R_OUTER - offset[1]),
                  displayInline=False)
d2.width = f"{2*R_OUTER + 2*offset[0]}mm"
d2.height = f"{2*R_OUTER + 2*offset[1]}mm"
d2.set_pixel_scale(1)
r2 = draw.Circle(0, 0, r=R_OUTER, fill='none', stroke='red', stroke_width="0.1")
d2.append(r2)
r3 = draw.Circle(0, 0, r=R_INNER, fill='none', stroke='red', stroke_width="0.1")
d2.append(r3)


for x, y in enlarged_dots:
    delta_y = random.uniform(DOT_DIAMETER_MM * 2, DOT_DIAMETER_MM * 5)
    delta_x = random.uniform(DOT_DIAMETER_MM * 1, DOT_DIAMETER_MM * 2)

    polar_x, polar_y = translate_to_polar_offset(x + delta_x, y, HEIGHT_MM / 2, R_INNER)
    polar_x2, polar_y2 = translate_to_polar_offset(x, y - delta_y, HEIGHT_MM / 2, R_INNER)
    polar_x3, polar_y3 = translate_to_polar_offset(x, y + delta_y, HEIGHT_MM / 2, R_INNER)


    p = draw.Path(stroke=ENGRAVE_COLOR, fill='none', stroke_width=0.1)
    d2.append(p.M(polar_x2, polar_y2).Q(polar_x, polar_y, polar_x3, polar_y3))
    d2.append(c)


filename_polar_enlarged = get_timestamped_filename('polar_enlarged')
d2.save_svg(filename_polar_enlarged)
# svg_to_fixed_width_html_image(SVG(filename_polar_enlarged).data.encode('ascii'), width="100%")


In [ ]:
## Original curve generation

# # Now use enlarged_dots with translate_to_polar_offset in the drawing
# d2 = draw.Drawing(2 * R_OUTER + 2 * offset[0],
#                   2 * R_OUTER + 2 * offset[1],
#                   origin=(-R_OUTER - offset[0], -R_OUTER - offset[1]),
#                   displayInline=False)
d2.width = f"{2*R_OUTER + 2*offset[0]}mm"
d2.height = f"{2*R_OUTER + 2*offset[1]}mm"
# d2.set_pixel_scale(1)
# r2 = draw.Circle(0, 0, r=R_OUTER, fill='none', stroke='red', stroke_width="0.1")
# d2.append(r2)
# r3 = draw.Circle(0, 0, r=R_INNER, fill='none', stroke='red', stroke_width="0.1")
# d2.append(r3)


# for x, y in enlarged_dots:
#     delta_y = random.uniform(DOT_DIAMETER_MM * 2, DOT_DIAMETER_MM * 5)

#     all_points_on_line_x = np.full(10, x)
#     all_points_on_line_y = np.linspace(y, y - delta_y, 10)

#     print(all_points_on_line_x)
#     print(all_points_on_line_y)

#     all_points_polar_x = []
#     all_points_polar_y = []

#     for i in range(10):
#         point_x = all_points_on_line_x[i]
#         point_y = all_points_on_line_y[i]
#         polar_x, polar_y = translate_to_polar_offset(point_x, point_y, HEIGHT_MM / 2, R_INNER)
#         all_points_polar_x.append(polar_x)
#         all_points_polar_y.append(polar_y)

#     xy = [item for sublist in zip(all_points_polar_x, all_points_polar_y) for item in sublist]
#     d2.append(dw.Lines(*xy, stroke=ENGRAVE_COLOR, stroke_width=0.05, fill='none'))


# filename_polar_curve = get_timestamped_filename('polar_curve')
# d2.save_svg(filename_polar_curve)
# svg_to_fixed_width_html_image(SVG(filename_polar_curve).data.encode('ascii'), width="100%")


In [ ]:
# Draw thick concentric circular arc patterns using enlarged_dots and translate_to_polar_offset with INNER/OUTER_OFFSET_MM protection
d2 = draw.Drawing(2 * R_OUTER + 2 * offset[0],
                  2 * R_OUTER + 2 * offset[1],
                  origin=(-R_OUTER - offset[0], -R_OUTER - offset[1]),
                  displayInline=False)
d2.width = f"{2*R_OUTER + 2*offset[0]}mm"
d2.height = f"{2*R_OUTER + 2*offset[1]}mm"
d2.set_pixel_scale(1)
r2 = draw.Circle(0, 0, r=R_OUTER, fill='none', stroke='red', stroke_width="0.1")
d2.append(r2)
r3 = draw.Circle(0, 0, r=R_INNER, fill='none', stroke='red', stroke_width="0.1")
d2.append(r3)

r_inner_limit = R_INNER + INNER_OFFSET_MM
r_outer_limit = R_OUTER - OUTER_OFFSET_MM
r_span = R_OUTER - R_INNER - INNER_OFFSET_MM - OUTER_OFFSET_MM

# Scale base stroke width proportionally with R_OUTER (baseline R_OUTER = 15.0)
scale_factor = R_OUTER / 60

for x, y in enlarged_dots:
    # Calculate radius and center angle on polar ring
    r_center = R_INNER + INNER_OFFSET_MM + (y / (HEIGHT_MM / 2)) * r_span
    theta_center = (x / (R_OUTER - R_INNER)) * 2 * np.pi

    # Arc length span along tangential direction
    delta_x = random.uniform(DOT_DIAMETER_MM * 5, DOT_DIAMETER_MM * 25)
    delta_theta = delta_x / r_center

    theta_start = theta_center - delta_theta / 2.0
    theta_end = theta_center + delta_theta / 2.0

    thick_stroke_width = random.uniform(0.1, 0.5) * scale_factor

    # Ensure stroke width maintains INNER_OFFSET_MM and OUTER_OFFSET_MM from red circles
    max_half_stroke = min(r_center - r_inner_limit, r_outer_limit - r_center)
    if max_half_stroke <= 0:
        continue

    thick_stroke_width = min(thick_stroke_width, 2 * max_half_stroke)

    arc_shape = create_thick_circular_arc(r_center, theta_start, theta_end, width=thick_stroke_width, fill_color=ENGRAVE_COLOR, cap='round')
    d2.append(arc_shape)


filename_thick_polar_curve = get_timestamped_filename('thick_polar_curve')
d2.save_svg(filename_thick_polar_curve)
svg_to_fixed_width_html_image(SVG(filename_thick_polar_curve).data.encode('ascii'), width="100%")


In [ ]:
def cap_value(value, min_value, max_value):
    return max(min_value, min(value, max_value))

In [ ]:
# # Now use enlarged_dots with translate_to_polar_offset in the drawing
# d2 = draw.Drawing(2 * R_OUTER + 2 * offset[0],
#                   2 * R_OUTER + 2 * offset[1],
#                   origin=(-R_OUTER - offset[0], -R_OUTER - offset[1]),
#                   displayInline=False)
d2.width = f"{2*R_OUTER + 2*offset[0]}mm"
d2.height = f"{2*R_OUTER + 2*offset[1]}mm"
# d2.set_pixel_scale(1)
# r2 = draw.Circle(0, 0, r=R_OUTER, fill='none', stroke='red', stroke_width="0.1")
# d2.append(r2)
# r3 = draw.Circle(0, 0, r=R_INNER, fill='none', stroke='red', stroke_width="0.1")
# d2.append(r3)


# for x, y in enlarged_dots:
#     delta_y = random.uniform(DOT_DIAMETER_MM * 1, DOT_DIAMETER_MM * 15)
#     delta_x = random.uniform(DOT_DIAMETER_MM * 1, DOT_DIAMETER_MM * 15)

#     polar_x, polar_y = translate_to_polar_offset(x, y, HEIGHT_MM / 2, R_INNER)
#     polar_x2, polar_y2 = translate_to_polar_offset(x - delta_y, y, HEIGHT_MM / 2, R_INNER)

#     c = draw.Line(polar_x, polar_y, polar_x2, polar_y2, stroke=ENGRAVE_COLOR, stroke_width="0.5")
#     d2.append(c)


# filename_digital_patch = get_timestamped_filename('digital_patch')
# d2.save_svg(filename_digital_patch)
# svg_to_fixed_width_html_image(SVG(filename_digital_patch).data.encode('ascii'), width="100%")


In [ ]:
# Draw squares/rectangles aligned with the radial direction of the circle with INNER/OUTER_OFFSET_MM protection
d2 = draw.Drawing(2 * R_OUTER + 2 * offset[0],
                  2 * R_OUTER + 2 * offset[1],
                  origin=(-R_OUTER - offset[0], -R_OUTER - offset[1]),
                  displayInline=False)
d2.width = f"{2*R_OUTER + 2*offset[0]}mm"
d2.height = f"{2*R_OUTER + 2*offset[1]}mm"
d2.set_pixel_scale(1)
r2 = draw.Circle(0, 0, r=R_OUTER, fill='none', stroke='red', stroke_width="0.1")
d2.append(r2)
r3 = draw.Circle(0, 0, r=R_INNER, fill='none', stroke='red', stroke_width="0.1")
d2.append(r3)

r_inner_limit = R_INNER + INNER_OFFSET_MM
r_outer_limit = R_OUTER - OUTER_OFFSET_MM

for x, y in enlarged_dots:
    polar_x, polar_y = translate_to_polar_offset(x, y, HEIGHT_MM / 2, R_INNER)
    r_center = np.hypot(polar_x, polar_y)

    # Maximum allowed radial half-length
    max_radial_half = min(r_center - r_inner_limit, r_outer_limit - r_center)
    if max_radial_half <= 0:
        continue

    # Dimensions along radial direction and tangential direction
    side_radial = random.uniform(DOT_DIAMETER_MM * 1, DOT_DIAMETER_MM * 10)
    side_tangential = random.uniform(DOT_DIAMETER_MM * 1, DOT_DIAMETER_MM * 5)

    # Cap radial side length to stay within offset limits
    side_radial = min(side_radial, 2 * max_radial_half)

    # Cap tangential side length so outer vertices do not cross outer offset limit
    r_outer_edge = r_center + side_radial / 2.0
    rem_dist_sq = max(0.0, r_outer_limit**2 - r_outer_edge**2)
    max_tangential_half = np.sqrt(rem_dist_sq)
    side_tangential = min(side_tangential, 2 * max_tangential_half)

    if side_radial <= 0 or side_tangential <= 0:
        continue

    # Calculate the radial angle in degrees from origin (0, 0)
    angle_deg = np.degrees(np.arctan2(polar_y, polar_x))

    # Draw rectangle/square centered at (polar_x, polar_y) rotated to align with radius
    c = draw.Rectangle(polar_x - side_radial / 2, polar_y - side_tangential / 2,
                       side_radial, side_tangential,
                       fill=ENGRAVE_COLOR, stroke='none',
                       transform=f'rotate({angle_deg}, {polar_x}, {polar_y})')
    d2.append(c)


filename_polar_squares = get_timestamped_filename('polar_squares')
d2.save_svg(filename_polar_squares)
svg_to_fixed_width_html_image(SVG(filename_polar_squares).data.encode('ascii'), width="100%")


In [ ]:
import ipywidgets as widgets
from IPython.display import display

# Scale slider parameters dynamically based on DOT_DIAMETER_MM and R_OUTER (baseline R_OUTER = 15.0)
scale_factor = R_OUTER / 60 if 'R_OUTER' in globals() else 1.0
base_dot_dia = DOT_DIAMETER_MM if 'DOT_DIAMETER_MM' in globals() else 0.1

init_min_dia = round(base_dot_dia * 0.5 * scale_factor, 2)
init_max_dia = round(base_dot_dia * 16.0 * scale_factor, 2)
min_slider_val = max(0.01, round(base_dot_dia * 0.1 * scale_factor, 2))
max_slider_val = round(base_dot_dia * 40.0 * scale_factor, 2)
step_val = max(0.01, round(0.01 * scale_factor, 2))

# Slider to adjust the dot diameter range dynamically (in mm)
dot_diameter_range_slider = widgets.FloatRangeSlider(
    value=[init_min_dia, init_max_dia],
    min=min_slider_val,
    max=max_slider_val,
    step=step_val,
    description='Dot Dia Range (mm):',
    continuous_update=False,
    readout_format='.2f'
)
display(dot_diameter_range_slider)


In [ ]:
# Draw dots of different sizes with translate_to_polar_offset in the drawing
if 'dot_diameter_range_slider' in globals():
    min_dia, max_dia = dot_diameter_range_slider.value
else:
    min_dia, max_dia = 0.05, 0.30

d2 = draw.Drawing(2 * R_OUTER + 2 * offset[0],
                  2 * R_OUTER + 2 * offset[1],
                  origin=(-R_OUTER - offset[0], -R_OUTER - offset[1]),
                  displayInline=False)
d2.width = f"{2*R_OUTER + 2*offset[0]}mm"
d2.height = f"{2*R_OUTER + 2*offset[1]}mm"
d2.set_pixel_scale(1)
r2 = draw.Circle(0, 0, r=R_OUTER, fill='none', stroke='red', stroke_width="0.1")
d2.append(r2)
r3 = draw.Circle(0, 0, r=R_INNER, fill='none', stroke='red', stroke_width="0.1")
d2.append(r3)

r_inner_limit = R_INNER + INNER_OFFSET_MM
r_outer_limit = R_OUTER - OUTER_OFFSET_MM

for x, y in enlarged_dots:
    # Generate initial random radius from slider diameter range
    r_dot = random.uniform(min_dia / 2.0, max_dia / 2.0)

    polar_x, polar_y = translate_to_polar_offset(x, y, HEIGHT_MM / 2, R_INNER)

    # Calculate distance from origin (0, 0)
    r_center = np.hypot(polar_x, polar_y)

    # Ensure the dot maintains at least INNER_OFFSET_MM and OUTER_OFFSET_MM distance from red circles
    max_allowed_r = min(r_center - r_inner_limit, r_outer_limit - r_center)

    if max_allowed_r <= 0:
        continue  # Skip if center is outside allowed region

    # Cap the dot radius to stay strictly within offset boundaries
    r_dot = min(r_dot, max_allowed_r)

    c = draw.Circle(polar_x, polar_y, r=r_dot, fill=ENGRAVE_COLOR, stroke='none')
    d2.append(c)


filename_polar_dots = get_timestamped_filename('polar_dots')
d2.save_svg(filename_polar_dots)
svg_to_fixed_width_html_image(SVG(filename_polar_dots).data.encode('ascii'), width="100%")


In [ ]:
# Draw dots with a strong distance-dependent diameter gradient (much larger dots near center, smaller near outer edge)
if 'dot_diameter_range_slider' in globals():
    min_dia, max_dia = dot_diameter_range_slider.value
else:
    min_dia, max_dia = 0.05, 0.30

d2 = draw.Drawing(2 * R_OUTER + 2 * offset[0],
                  2 * R_OUTER + 2 * offset[1],
                  origin=(-R_OUTER - offset[0], -R_OUTER - offset[1]),
                  displayInline=False)
d2.width = f"{2*R_OUTER + 2*offset[0]}mm"
d2.height = f"{2*R_OUTER + 2*offset[1]}mm"
d2.set_pixel_scale(1)
r2 = draw.Circle(0, 0, r=R_OUTER, fill='none', stroke='red', stroke_width="0.1")
d2.append(r2)
r3 = draw.Circle(0, 0, r=R_INNER, fill='none', stroke='red', stroke_width="0.1")
d2.append(r3)

r_inner_limit = R_INNER + INNER_OFFSET_MM
r_outer_limit = R_OUTER - OUTER_OFFSET_MM

for x, y in enlarged_dots:
    polar_x, polar_y = translate_to_polar_offset(x, y, HEIGHT_MM / 2, R_INNER)

    # Calculate distance from origin (0, 0)
    r_center = np.hypot(polar_x, polar_y)

    # Ensure the dot maintains at least INNER_OFFSET_MM and OUTER_OFFSET_MM distance from red circles
    max_allowed_r = min(r_center - r_inner_limit, r_outer_limit - r_center)

    if max_allowed_r <= 0:
        continue  # Skip if center is outside allowed region

    # Normalized distance t from inner boundary (0: near center, 1: near outer boundary)
    t = cap_value((r_center - r_inner_limit) / (r_outer_limit - r_inner_limit), 0.0, 1.0)

    # Strong non-linear distance factor (1.0 at inner boundary -> 0.0 at outer boundary)
    dist_factor = (1.0 - t) ** 1.5

    # Add slight random fluctuation (+/- 15%) while maintaining strong gradient
    noise = random.uniform(-0.15, 0.15)
    size_factor = cap_value(dist_factor + noise, 0.0, 1.0)

    r_dot = (min_dia + size_factor * (max_dia - min_dia)) / 2.0

    # Cap the dot radius to stay strictly within offset boundaries
    r_dot = min(r_dot, max_allowed_r)

    c = draw.Circle(polar_x, polar_y, r=r_dot, fill=ENGRAVE_COLOR, stroke='none')
    d2.append(c)


filename_polar_dots_weighted = get_timestamped_filename('polar_dots_weighted')
d2.save_svg(filename_polar_dots_weighted)
svg_to_fixed_width_html_image(SVG(filename_polar_dots_weighted).data.encode('ascii'), width="100%")


In [ ]:
# Generate test.svg with a 2cm (20mm) red square containing a circle, a thick closed arc, and a square
box_size = 20.0  # 2 cm = 20 mm

d_test = draw.Drawing(box_size, box_size, origin=(-box_size / 2, -box_size / 2), displayInline=False)
d_test.width = f"{box_size}mm"
d_test.height = f"{box_size}mm"
d_test.set_pixel_scale(1)

# 1. Red 2 cm (20 mm) square outer boundary
r_boundary = draw.Rectangle(-box_size / 2, -box_size / 2, box_size, box_size, fill='none', stroke='red', stroke_width="0.1")
d_test.append(r_boundary)

# 2. Green Circle (left position x = -5 mm, y = 0)
circle_shape = draw.Circle(-5.0, 0.0, r=2.0, fill=ENGRAVE_COLOR, stroke='none')
d_test.append(circle_shape)

# 3. Green Arc (center position x = 0 mm, y = 0) - Closed 2D shape with 0.5mm physical width and convex protruding round cap
arc_shape = create_thick_bezier_path(p0=(0.0, -4.0), p1=(3.0, 0.0), p2=(0.0, 4.0), width=0.5, fill_color=ENGRAVE_COLOR, cap='round')
d_test.append(arc_shape)

# 4. Green Square (right position x = +5 mm, y = 0)
square_shape = draw.Rectangle(5.0 - 1.75, 0.0 - 1.75, 3.5, 3.5, fill=ENGRAVE_COLOR, stroke='none')
d_test.append(square_shape)

filename_test = 'test.svg'
d_test.save_svg(filename_test)
print(f"Saved {filename_test} successfully (2cm x 2cm) with convex arc shape.")
